# Kiwi — reconocer y devolver varias herramientas

Flujo principal: fotografía → Grounding DINO → recortes → Qwen3-VL-4B → ubicaciones de todas las herramientas aceptadas. Ejecutar 1–5c en orden y abrir Gradio (6) o REST (6b). La Raspberry Pi verifica cada ubicación por separado. GPU recomendada; CPU usa Qwen NF4 y tarda varios minutos. No hay filtros nuevos de fondo.


## 1. Instalar dependencias
Si Colab pide reiniciar la sesión después de instalar, reiniciar y continuar con la celda 2. No reinstalamos PyTorch: usamos el de Colab.

In [ ]:
%pip install -q -U sentence-transformers transformers accelerate bitsandbytes gradio


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Categorías y ejecución
Las categorías deben coincidir con las de la Pi. EmbeddingGemma es una verificación adicional por recorte, desactivada por defecto; DINO y Qwen siempre son el flujo principal.


In [ ]:
import os
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
import time
import numpy as np
import torch
from PIL import Image, ImageOps
from sentence_transformers import SentenceTransformer

MODEL_ID = 'google/embeddinggemma-2'
CATEGORIES = {
    'soldering_station': 'An electronics soldering and hot air rework station with a control unit and iron holder',
    'digital_multimeter': 'A handheld digital multimeter with a screen and rotary selector',
    'bench_power_supply': 'A laboratory bench DC power supply with output terminals and controls',
    'servo_motor': 'A small blue micro servo motor with a three-wire cable',
    'oscilloscope': 'A digital oscilloscope with a waveform screen and input connectors',
    'function_generator': 'A laboratory function generator with frequency controls and signal outputs',
    'variable_capacitor': 'A laboratory variable parallel plate capacitor with circular metal plates on a base',
    'capacitors': 'A small cylindrical electrolytic capacitor with two wire leads',
    'power_adapter': 'A rectangular AC DC power adapter with an integrated output cable',
    'multimeter_box': 'A yellow orange cardboard retail box for an AstroAI digital multimeter',
    'solder': 'An electric soldering iron with a handle and pointed metal tip',
    'otros': 'An everyday object that is not one of the laboratory tools, such as household items, hand tools or bare circuit boards',
}
MIN_SIMILARITY = None
MIN_MARGIN = None
LOCATION_BY_CATEGORY = {}  # Ejemplo al definir posiciones reales: {'digital_multimeter': 2}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
VLM_CHECK_WITH_REFERENCES = False
names = list(CATEGORIES)
model = None
if VLM_CHECK_WITH_REFERENCES:
    model = SentenceTransformer(MODEL_ID, device=device,
        config_kwargs={'audio_config': None}, model_kwargs={'torch_dtype': dtype})
    model.eval()
print('Dispositivo:', device, '| verificación por referencias:', VLM_CHECK_WITH_REFERENCES)


## 3. Decidir cuándo aceptar una comparación
Rechazamos resultados débiles o ambiguos. Si faltan umbrales de validación, mostramos el candidato sin autorizar la señalización.

In [ ]:
import math

def decide(scores, minimum=None, margin=None):
    ranking = sorted(
        ((name, float(score)) for name, score in scores.items() if math.isfinite(float(score))),
        key=lambda item: item[1], reverse=True,
    )
    result = {'candidate': None, 'recognized': False, 'similarity': None,
              'margin': None, 'reason': 'No hay suficientes categorías válidas.'}
    if not ranking:
        return result
    result.update(candidate=ranking[0][0], similarity=ranking[0][1])
    if len(ranking) < 2:
        return result
    result['margin'] = ranking[0][1] - ranking[1][1]
    if minimum is None or margin is None:
        result['reason'] = 'Modo de prueba: umbrales sin calibrar.'
    elif ranking[0][1] < minimum:
        result['reason'] = 'Similitud insuficiente: objeto desconocido o captura inadecuada.'
    elif result['margin'] < margin:
        result['reason'] = 'Comparación ambigua entre las categorías principales.'
    else:
        result.update(recognized=True, reason='Comparación aceptada con los umbrales configurados.')
    return result

## 4. Preparar recortes y verificación adicional
Las referencias se usan solo cuando `VLM_CHECK_WITH_REFERENCES=True`; no sustituyen DINO/Qwen.


In [ ]:
names = list(CATEGORIES)
text_vectors = model.encode(list(CATEGORIES.values()), normalize_embeddings=True, convert_to_numpy=True) if model is not None else None
reference_vectors = None
reference_labels = []

def prepare_image(image):
    image = ImageOps.exif_transpose(image).convert('RGB')
    image.thumbnail((1024, 1024))
    return image

def identify_embeddings(image):
    if image is None:
        return {'recognized': False, 'candidate': None, 'location_id': None,
                'reason': 'Sube una fotografía.'}
    if model is None:
        raise RuntimeError('Activa la verificación por referencias en la sección 2.')
    start = time.perf_counter()
    vector = model.encode({'image': prepare_image(image)}, normalize_embeddings=True, convert_to_numpy=True)
    if reference_vectors is None:
        raw_scores = np.asarray(text_vectors @ vector).reshape(-1)
        scores = dict(zip(names, raw_scores))
        mode = 'image_to_text'
    else:
        raw_scores = np.asarray(reference_vectors @ vector).reshape(-1)
        scores = {name: max(float(score) for label, score in zip(reference_labels, raw_scores) if label == name)
                  for name in names}
        mode = 'image_to_image'
    result = decide(scores, MIN_SIMILARITY, MIN_MARGIN)
    result['mode'] = mode
    result['location_id'] = LOCATION_BY_CATEGORY.get(result['candidate']) if result['recognized'] else None
    result['elapsed_seconds'] = round(time.perf_counter() - start, 3)
    result['ranking'] = [{'object': name, 'similarity': round(float(score), 4)}
                         for name, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)]
    return result

print('Comparación preparada. Las referencias y los umbrales se pueden ajustar sin recargar el modelo.')
identify = identify_embeddings  # solo para reutilizar el verificador dentro de Qwen


## 5. Referencias adicionales desde Drive
Desactivadas por defecto. Para verificarlas, activar `VLM_CHECK_WITH_REFERENCES` en 2 y cargar `references.zip` de Drive. Las correcciones de la Pi guardan recortes individuales.


In [ ]:
LOAD_REFERENCE_ZIP = VLM_CHECK_WITH_REFERENCES
if LOAD_REFERENCE_ZIP:
    import io
    import zipfile
    from pathlib import PurePosixPath, Path
    zip_path = Path('/content/drive/MyDrive/Kiwi/references.zip')
    if not zip_path.is_file():
        available = [p.name for p in zip_path.parent.glob('*.zip')]
        raise FileNotFoundError(f'No existe {zip_path}. ZIPs disponibles en Kiwi: {available}')
    zip_name, zip_data = zip_path.name, zip_path.read_bytes()
    print('Cargando referencias desde Drive:', zip_name)
    if len(zip_data) > 100 * 1024 * 1024:
        raise ValueError('El ZIP debe pesar menos de 100 MB.')
    vectors, labels = [], []
    with zipfile.ZipFile(io.BytesIO(zip_data)) as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) > 500 * 1024 * 1024:
            raise ValueError('El contenido del ZIP supera 500 MB.')
        for member in members:
            path = PurePosixPath(member.filename)
            if member.is_dir() or path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:
                continue
            if '__MACOSX' in path.parts or path.name.startswith('.'):
                continue
            label = path.parent.name
            if label not in CATEGORIES:
                raise ValueError(f'Carpeta desconocida: {label}. Usar los nombres de CATEGORIES.')
            if member.file_size > 15 * 1024 * 1024:
                raise ValueError(f'Foto demasiado grande: {path.name}. Máximo 15 MB.')
            with Image.open(io.BytesIO(archive.read(member))) as photo:
                vector = model.encode({'image': prepare_image(photo)}, normalize_embeddings=True, convert_to_numpy=True)
            vectors.append(vector)
            labels.append(label)
    missing = set(CATEGORIES) - set(labels)
    if missing:
        raise ValueError('Faltan referencias de: ' + ', '.join(sorted(missing)))
    reference_vectors = np.stack(vectors)
    reference_labels = labels
    MIN_SIMILARITY = None
    MIN_MARGIN = None
    print('Referencias cargadas:', {name: labels.count(name) for name in names})
    print('Se restablecieron los umbrales: calibrar para el nuevo modo.')
else:
    print('Sin cambios en las referencias. Modo actual:', 'fotografías' if reference_vectors is not None else 'descripciones')

## 5b. Qwen3-VL-4B: clasificar cada recorte
Rechaza objetos ajenos o ambiguos. CPU usa NF4 y menor resolución; sus puntuaciones pueden cambiar respecto a GPU. No son probabilidades calibradas.


In [ ]:
# Kiwi · reconocimiento con un modelo de visión y lenguaje (VLM), sin fotos de referencia.
#
# Se ejecuta en el notebook después de las secciones 1–4 (usa CATEGORIES, decide e identify):
#     exec(open('/content/kiwi_vlm.py').read())
# Reemplaza `identify`, así que la API REST de la sección 6b y la página de Gradio pasan a usar el VLM
# sin volver a ejecutarlas. Para volver a EmbeddingGemma: identify = identify_embeddings
#
# El VLM recibe la foto y una pregunta de opción múltiple con las categorías; se leen las probabilidades
# de cada letra de respuesta. `similarity` pasa a ser esa probabilidad (0–1) y `margin` la diferencia con
# la segunda opción. La opción "ninguna" se devuelve como la categoría `otros`.
import time
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

VLM_ID = 'Qwen/Qwen3-VL-4B-Instruct'
VLM_MIN_PROBABILITY = 0.90   # recognized=True solo por encima de esta probabilidad...
VLM_MIN_MARGIN = 0.50        # ...y con esta ventaja sobre la segunda opción
# Doble verificación: si la categoría tiene fotos de referencia (sección 5) y EmbeddingGemma elige otra
# herramienta, no se acepta. Evita LEDs equivocados con fotos ambiguas (p. ej. equipos vistos por detrás).
VLM_CHECK_WITH_REFERENCES = globals().get("VLM_CHECK_WITH_REFERENCES", False)
OTHER = 'otros'

if 'identify_embeddings' not in globals():
    identify_embeddings = identify   # el reconocimiento por fotos de referencia sigue disponible

if 'vlm_model' not in globals():
    cpu = not torch.cuda.is_available()
    print('Cargando', VLM_ID, '| CPU NF4' if cpu else '| GPU', flush=True)
    vlm_processor = AutoProcessor.from_pretrained(VLM_ID,
        min_pixels=64 * 32 * 32 if cpu else 256 * 28 * 28,
        max_pixels=128 * 32 * 32 if cpu else 640 * 28 * 28)
    kwargs = {'dtype': torch.float32 if cpu else torch.float16,
              'device_map': 'cpu' if cpu else 'auto', 'attn_implementation': 'sdpa'}
    if cpu:
        kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True,
            bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float32)
        torch.set_num_threads(2)
    vlm_model = AutoModelForImageTextToText.from_pretrained(VLM_ID, **kwargs).eval()
    print('Qwen cargado; memoria de pesos: %.1f GB' % (vlm_model.get_memory_footprint() / 1e9))

# Descripciones para el VLM: rasgos del panel frontal que distinguen equipos parecidos por fuera.
# Las categorías sin entrada aquí usan su texto de CATEGORIES.
VLM_DESCRIPTIONS = {
    'oscilloscope': 'Oscilloscope: bench instrument whose front has a large screen with a grid showing waveforms, '
                    'many knobs and BNC probe inputs labeled CH1/CH2',
    'function_generator': 'Function generator: bench instrument whose front has a numeric frequency display, '
                          'waveform buttons (sine, square, triangle) and BNC signal output connectors',
    'bench_power_supply': 'Bench DC power supply: bench instrument whose front shows voltage (V) and current (A) '
                          'readouts, adjustment knobs and red/black banana output terminals',
    'soldering_station': 'Soldering station: control unit with a temperature display or dial, connected to a '
                         'soldering iron and an iron holder stand',
    'power_adapter': 'AC/DC power adapter: small black plastic brick with an attached cable, used to power a '
                     'device from the wall outlet',
    'digital_multimeter': 'Handheld digital multimeter: portable meter with an LCD screen, a large rotary selector '
                          'dial and probe jacks (COM, V, A)',
    'solder': 'Soldering iron: pen-shaped hand tool with a plastic handle and a long pointed metal tip',
}

VLM_OPTIONS = [name for name in CATEGORIES if name != OTHER] + [OTHER]
VLM_LETTERS = [chr(ord('A') + i) for i in range(len(VLM_OPTIONS))]
VLM_LETTER_IDS = [vlm_processor.tokenizer.encode(letter, add_special_tokens=False)[0] for letter in VLM_LETTERS]
assert len(set(VLM_LETTER_IDS)) == len(VLM_LETTERS), 'Cada letra debe ser un token distinto.'


def vlm_prompt():
    lines = [
        'You are checking which tool someone is returning to an electronics lab shelf.',
        'Look only at the main object in the photo; ignore hands, tables, cables in the background and other clutter.',
        'Many bench instruments are similar boxes: decide by what is visible on their front panel. If only the back, '
        'bottom or side of an instrument is visible and you cannot see its front panel, choose the last option.',
        'Options:',
    ]
    for letter, name in zip(VLM_LETTERS, VLM_OPTIONS):
        if name == OTHER:
            lines.append(f'{letter}. None of the above or cannot tell: the main object is something else (for '
                         'example a resistor, LED, loose wire, Arduino or other circuit board, pliers, screwdriver, '
                         'phone or household item), there is no clear object, or only the back/side of an '
                         'instrument is visible.')
        else:
            lines.append(f'{letter}. {VLM_DESCRIPTIONS.get(name, CATEGORIES[name])}.')
    lines.append('Answer with only the letter of the correct option.')
    return '\n'.join(lines)


@torch.inference_mode()
def vlm_scores(image):
    messages = [{'role': 'user', 'content': [{'type': 'image', 'image': image},
                                             {'type': 'text', 'text': vlm_prompt()}]}]
    inputs = vlm_processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors='pt').to(vlm_model.device)
    logits = vlm_model(**inputs).logits[0, -1].float()
    probabilities = torch.softmax(logits[VLM_LETTER_IDS], dim=0).tolist()
    return dict(zip(VLM_OPTIONS, probabilities))


def identify_vlm(image):
    if image is None:
        return {'recognized': False, 'candidate': None, 'location_id': None, 'reason': 'Sube una fotografía.'}
    start = time.perf_counter()
    scores = vlm_scores(prepare_image(image))
    ranking = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    candidate, probability = ranking[0]
    margin = probability - ranking[1][1]
    result = {'candidate': candidate, 'similarity': probability, 'margin': margin, 'mode': 'vlm',
              'model': VLM_ID, 'recognized': False}
    if candidate == OTHER:
        result['reason'] = 'No es una de las herramientas o no se ve su frente. Muestra el frente del objeto.'
    elif probability < VLM_MIN_PROBABILITY:
        result['reason'] = 'Confianza insuficiente: foto poco clara u objeto desconocido.'
    elif margin < VLM_MIN_MARGIN:
        result['reason'] = 'Duda entre dos herramientas. Repite la foto.'
    else:
        result.update(recognized=True, reason='Reconocido por el VLM.')
        if (VLM_CHECK_WITH_REFERENCES and globals().get('reference_vectors') is not None
                and candidate in set(reference_labels)):
            check = identify_embeddings(image)
            result['reference_candidate'] = check['candidate']
            result['reference_similarity'] = check['similarity']
            if check['candidate'] != candidate:
                result.update(recognized=False, reason='El VLM y las fotos de referencia no coinciden. '
                                                        'Muestra el frente de la herramienta.')
    result['location_id'] = LOCATION_BY_CATEGORY.get(candidate) if result['recognized'] else None
    result['elapsed_seconds'] = round(time.perf_counter() - start, 3)
    result['ranking'] = [{'object': name, 'similarity': round(p, 4)} for name, p in ranking]
    return result


print('Clasificación de recortes preparada:', VLM_ID)


## 5c. Grounding DINO: localizar todos los objetos
Cada recorte pasa por Qwen. DINO puede omitir objetos o marcar el fondo: esta integración conserva los umbrales del experimento.


In [ ]:
"""Reconocimiento principal: Grounding DINO localiza y Qwen clasifica recortes.

No sustituye identify ni modifica la API o las luces de la estación.
"""
import inspect
import math
import time
from PIL import ImageDraw, ImageOps

DETECTOR_ID = 'IDEA-Research/grounding-dino-tiny'
DETECTION_PROMPT = (
    'a soldering station. a digital multimeter. a laboratory power supply. '
    'a servo motor. an oscilloscope. a function generator. a variable capacitor. '
    'a capacitor. a power adapter. a yellow multimeter box. a soldering iron.'
)


def intersection_over_union(a, b):
    intersection = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    areas = [(x[2] - x[0]) * (x[3] - x[1]) for x in (a, b)]
    union = sum(areas) - intersection
    return intersection / union if union > 0 else 0


def select_detections(detections, size, limit=12, iou_threshold=.6):
    """Clamp boxes and remove near duplicates across detector labels."""
    width, height = size
    selected = []
    for item in sorted(detections, key=lambda x: x['score'], reverse=True):
        if not all(math.isfinite(v) for v in [item['score'], *item['box']]):
            continue
        x1, y1, x2, y2 = item['box']
        box = [max(0, min(width, math.floor(x1))), max(0, min(height, math.floor(y1))),
               max(0, min(width, math.ceil(x2))), max(0, min(height, math.ceil(y2)))]
        if box[2] <= box[0] or box[3] <= box[1]:
            continue
        if any(intersection_over_union(box, old['box']) > iou_threshold for old in selected):
            continue
        selected.append(dict(item, box=box))
        if len(selected) >= limit:
            break
    return selected


class MultiObjectRecognizer:
    def __init__(self, classify):
        import torch
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        self.classify = classify
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.processor = AutoProcessor.from_pretrained(DETECTOR_ID)
        self.detector = AutoModelForZeroShotObjectDetection.from_pretrained(DETECTOR_ID).to(self.device).eval()

    def detect(self, image, threshold=.25, text_threshold=.2, limit=12):
        import torch
        inputs = self.processor(images=image, text=DETECTION_PROMPT, return_tensors='pt').to(self.device)
        with torch.inference_mode():
            outputs = self.detector(**inputs)
        postprocess = self.processor.post_process_grounded_object_detection
        # Transformers versions use either threshold or box_threshold.
        threshold_name = 'threshold' if 'threshold' in inspect.signature(postprocess).parameters else 'box_threshold'
        result = postprocess(outputs, inputs.input_ids, **{threshold_name: float(threshold)},
                             text_threshold=float(text_threshold), target_sizes=[image.size[::-1]])[0]
        labels = result.get('text_labels', result.get('labels'))
        raw = [{'box': box.tolist(), 'score': float(score), 'label': str(label)}
               for box, score, label in zip(result['boxes'], result['scores'], labels)]
        return select_detections(raw, image.size, limit), len(raw)

    def locate(self, image, threshold=.25):
        if image is None:
            return {'image_size': None, 'objects': [], 'raw_detections': 0,
                    'detection_seconds': 0.0}
        image = ImageOps.exif_transpose(image).convert('RGB')
        start = time.perf_counter()
        detections, raw_count = self.detect(image, threshold)
        objects = [{'id': index, 'box': detection['box'],
                    'detector_label': detection['label'],
                    'detector_score': detection['score']}
                   for index, detection in enumerate(detections[:12], 1)]
        return {'image_size': list(image.size), 'objects': objects,
                'raw_detections': raw_count,
                'detection_seconds': round(time.perf_counter() - start, 3)}

    def classify_crop(self, image):
        if image is None:
            return {'recognized': False, 'candidate': None, 'location_id': None,
                    'reason': 'Sube una fotografía.'}
        image = ImageOps.exif_transpose(image).convert('RGB')
        return self.classify(image)

    def identify(self, image, threshold=.25):
        if image is None:
            return None, [], {'mode': 'grounding_dino_qwen', 'image_size': None, 'objects': [], 'reason': 'Sube una foto.'}
        image = ImageOps.exif_transpose(image).convert('RGB')
        start = time.perf_counter()
        located = self.locate(image, threshold)
        annotated = image.copy()
        draw = ImageDraw.Draw(annotated)
        objects, crops = [], []
        for detection in located['objects']:
            index = detection['id']
            box = detection['box']
            # A small border preserves context without including most neighbors.
            pad = max(2, round(min(box[2] - box[0], box[3] - box[1]) * .04))
            crop_box = [max(0, box[0] - pad), max(0, box[1] - pad),
                        min(image.width, box[2] + pad), min(image.height, box[3] + pad)]
            crop = image.crop(crop_box)
            result = self.classify_crop(crop)
            accepted = bool(result.get('recognized'))
            candidate = result.get('candidate')
            caption = f'{index}: {candidate or "desconocido"}' + ('' if accepted else ' (revisar)')
            color = 'green' if accepted else 'orange'
            draw.rectangle(box, outline=color, width=max(2, image.width // 400))
            draw.text((box[0], max(0, box[1] - 14)), caption, fill=color)
            crops.append((crop, caption))
            objects.append({'id': index, 'box': box, 'crop_box': crop_box,
                            'detector_label': detection['detector_label'],
                            'detector_score': detection['detector_score'],
                            'category': candidate, 'recognized': accepted, 'recognition': result})
        detection_seconds = located['detection_seconds']
        return annotated, crops, {'mode': 'grounding_dino_qwen', 'image_size': list(image.size), 'objects': objects, 'raw_detections': located['raw_detections'],
                                  'detection_seconds': detection_seconds,
                                  'elapsed_seconds': round(time.perf_counter() - start, 3),
                                  'limit': 12, 'reason': 'Sin objetos detectados.' if not objects else
                                  'Revisa objetos omitidos y componentes del mismo equipo.'}


# Alias para reproducir los experimentos CPU históricos.
MultiObjectProbe = MultiObjectRecognizer

import threading
model_lock = globals().get('model_lock') or threading.Lock()
multi_recognizer = MultiObjectRecognizer(identify_vlm)
print('Reconocedor principal listo: Grounding DINO + Qwen; máximo 12 recortes.')
def identify_current(image):
    if globals().get('multi_recognizer') is None or globals().get('vlm_model') is None:
        raise RuntimeError('Carga Qwen y Grounding DINO en las secciones 5b–5c.')
    with model_lock:
        return multi_recognizer.identify(image)

def identify_objects(image):
    return identify_current(image)[2]



## 6. Página Gradio para varias herramientas
La respuesta incluye foto anotada, recortes y JSON. Ejecutar 5b y 5c antes de abrirla. Usuario `kiwi` y contraseña de esta sesión.


In [ ]:
import getpass
import threading
import gradio as gr

model_lock = globals().get('model_lock') or threading.Lock()

if 'demo' in globals():
    demo.close()
APP_USER = 'kiwi'
APP_PASSWORD = getpass.getpass('Contraseña para la página y la API (mínimo 12 caracteres): ')
if len(APP_PASSWORD) < 12:
    raise ValueError('Usa al menos 12 caracteres y vuelve a ejecutar esta celda.')
demo = gr.Interface(
    fn=identify_current,
    inputs=gr.Image(type='pil', sources=['upload', 'webcam'], label='Fotografía de herramientas'),
    outputs=[gr.Image(label='Objetos localizados'), gr.Gallery(label='Recortes'), gr.JSON(label='Resultados')],
    title='Kiwi · Devolver herramientas',
    description='Sube una foto con varias herramientas. Revisa los cuadros y los objetos rechazados.',
    api_name='identify',
    flagging_mode='never',
    submit_btn='Identificar',
    clear_btn='Limpiar',
)
demo.queue(default_concurrency_limit=1, max_size=10)
demo.launch(share=True, auth=(APP_USER, APP_PASSWORD), debug=False)

## 6b. API REST para la Raspberry Pi (ngrok)
La estación de la Raspberry Pi (`pi/`) usa `POST /predict` para identificación manual y `POST /detect` más `POST /classify` para el escaneo continuo. `GET /health` anuncia las capacidades disponibles. Esta celda publica esas rutas con FastAPI y un túnel de ngrok. Puede funcionar junto con la página de Gradio de la sección 6 o sin ella.

Antes de ejecutarla, en **Secretos** (icono de llave) de Colab crear y habilitar para este notebook:
- `NGROK_AUTHTOKEN`: token de una cuenta gratuita de ngrok.
- `KIWI_API_KEY`: una clave larga inventada por el equipo. La misma va en la Pi como variable `KIWI_API_KEY`.

Copiar la URL que termina en `/predict` en el campo **Endpoint de Colab** de la pantalla táctil (o en `pi/config.yaml`). La Pi deriva `/detect` y `/classify` de esa base. La URL cambia en cada ejecución salvo que se use un dominio fijo de ngrok.

In [ ]:
%pip install -q pyngrok python-multipart

In [ ]:
import io
import json
import secrets
import threading
import uvicorn
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from google.colab import userdata
from pyngrok import ngrok

API_KEY = userdata.get('KIWI_API_KEY')
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
if not API_KEY or len(API_KEY) < 16:
    raise ValueError('Crea el secreto KIWI_API_KEY con al menos 16 caracteres.')
API_PORT = 8008
model_lock = globals().get('model_lock') or threading.Lock()
api = FastAPI(title='Kiwi API')

@api.get('/health')
def health():
    detector_ready = globals().get('multi_recognizer') is not None
    classifier_ready = detector_ready and globals().get('vlm_model') is not None
    capabilities = {'predict': classifier_ready, 'detect': detector_ready,
                    'classify': classifier_ready}
    return {'ok': classifier_ready, 'mode': 'grounding_dino_qwen', 'calibrated': False,
            'thresholds_configured': classifier_ready, 'categories': names,
            'capabilities': capabilities}

def _authorize(authorization):
    if not secrets.compare_digest(authorization or '', f'Bearer {API_KEY}'):
        raise HTTPException(401, 'Clave incorrecta.')

def _read_image(file):
    data = file.file.read(15 * 1024 * 1024 + 1)
    if len(data) > 15 * 1024 * 1024:
        raise HTTPException(413, 'La foto supera 15 MB.')
    try:
        image = Image.open(io.BytesIO(data))
        image.load()
        return image
    except Exception as exc:
        raise HTTPException(400, 'El archivo no es una imagen válida.') from exc

def _parse_correlation(value):
    try:
        result = json.loads(value)
    except (TypeError, ValueError) as exc:
        raise HTTPException(400, 'La correlación debe ser un objeto JSON.') from exc
    if not isinstance(result, dict):
        raise HTTPException(400, 'La correlación debe ser un objeto JSON.')
    return result

def _process_scan(file, correlation, authorization, operation):
    _authorize(authorization)
    correlation = _parse_correlation(correlation)
    image = _read_image(file)
    if globals().get('multi_recognizer') is None:
        raise HTTPException(503, 'Carga Grounding DINO en la sección 5c.')
    if operation == 'classify' and globals().get('vlm_model') is None:
        raise HTTPException(503, 'Carga Qwen en la sección 5b.')
    try:
        with model_lock:
            if operation == 'detect':
                result = multi_recognizer.locate(image)
            elif operation == 'classify':
                result = multi_recognizer.classify_crop(image)
            else:
                raise HTTPException(404, 'Operación de escaneo desconocida.')
    except RuntimeError as exc:
        raise HTTPException(503, str(exc)) from exc
    return {'correlation': correlation, 'result': result}

@api.post('/predict')
def predict(file: UploadFile = File(...), authorization: str | None = Header(None)):
    _authorize(authorization)
    image = _read_image(file)
    try:
        return identify_objects(image)
    except RuntimeError as exc:
        raise HTTPException(503, str(exc)) from exc

@api.post('/detect')
def detect(file: UploadFile = File(...), correlation: str = Form(...),
           authorization: str | None = Header(None)):
    return _process_scan(file, correlation, authorization, 'detect')

@api.post('/classify')
def classify(file: UploadFile = File(...), correlation: str = Form(...),
             authorization: str | None = Header(None)):
    return _process_scan(file, correlation, authorization, 'classify')

if 'api_server' in globals():
    api_server.should_exit = True
    ngrok.kill()
api_server = uvicorn.Server(uvicorn.Config(api, host='127.0.0.1', port=API_PORT, log_level='warning'))
threading.Thread(target=api_server.run, daemon=True).start()
ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(API_PORT, 'http')
print('Endpoint para la Raspberry Pi:', tunnel.public_url + '/predict')
print('Escaneo continuo:', tunnel.public_url + '/detect', 'y', tunnel.public_url + '/classify')
print('Modo:', health()['mode'], '| capacidades:', health()['capabilities'])

## 7. Consumir los resultados
REST `/predict` devuelve `objects` e `image_size`. Gradio `/identify` devuelve imagen anotada, galería y JSON: usar la tercera salida para la Pi. La Pi enciende todas las ubicaciones aceptadas y verifica presencia por ubicación. Con sensores desactivados no declara devolución verificada.

Probar varias herramientas en una foto real, categorías repetidas, objetos ajenos, devolución parcial y sensores equivocados antes de la demostración.
